# SignBridge M4 — WLASL isolated-word BiGRU (Colab)

This notebook trains the agreed M4 isolated-word classifier from the M3 sharded landmark dataset. It expects the M3 artifacts (`dataset.index.json`, `dataset.manifest.jsonl`, and `shard-*.npz`) and **does not rebuild videos or landmarks**.

Design contract:
- each sample is one WLASL isolated word;
- each frame is 110 points × 3 coordinates (330 input values);
- the manifest preserves the official WLASL `train`, `val`, and `test` splits;
- M3 right-padding is trimmed and each sequence is linearly resampled to 60 temporal positions;
- a small two-layer bidirectional GRU classifies the normalized sequence;
- the best checkpoint, metrics, and the exact label mapping are exported together.

Run the cells from top to bottom. The first configuration cell is the only required path/configuration edit.

In [ ]:
# Colab setup. PyTorch is normally preinstalled in a GPU runtime.
# Uncomment the next line only if this runtime is missing a required package.
# !pip -q install scikit-learn matplotlib

import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 1. Paths and configuration

Put the M3 output directory in Drive or in `/content`, then update `DATA_DIR`. A sharded M3 directory must contain the index, JSONL manifest, and all shard files listed by the index. Set `MAX_SAMPLES_PER_SPLIT` to a small integer (for example, 64) for a quick smoke run; leave it as `None` for the full benchmark.

In [ ]:
# Optional Drive mount for a dataset stored at /content/drive/MyDrive/.
# from google.colab import drive
# drive.mount('/content/drive')

DATA_DIR = Path('/content/signbridge_wlasl_m3')
# Example Drive path: DATA_DIR = Path('/content/drive/MyDrive/signbridge_wlasl_m3')
OUTPUT_DIR = Path('/content/signbridge_m4_bigru')

SEED = 42
MAX_SAMPLES_PER_SPLIT = None  # set to an integer for a smoke run
TARGET_FRAMES = 60
FEATURE_SHAPE = (110, 3)
INPUT_SIZE = int(np.prod(FEATURE_SHAPE))  # 330
BATCH_SIZE = 64
EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_SIZE = 128
GRU_LAYERS = 2
DROPOUT = 0.25
PATIENCE = 8
GRAD_CLIP = 1.0
PREPROCESSING_VERSION = 'temporal-normalize-nonzero-v1'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert INPUT_SIZE == 330
print('M3 input:', DATA_DIR)
print('M4 output:', OUTPUT_DIR)

In [ ]:
# Reproducibility and device selection.
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f'M3 data directory not found: {DATA_DIR}. Update DATA_DIR and rerun this cell.'
    )

## 2. M3 sharded loader

The loader reads every manifest row and uses its `subset`, `shard`, and `row_in_shard` fields. It therefore keeps the official WLASL split assignment instead of randomly repartitioning extracted samples. `MAX_SAMPLES_PER_SPLIT` limits each split independently and is useful only for smoke tests.

In [ ]:
def _read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def load_m3_shards(data_dir, max_samples_per_split=None):
    data_dir = Path(data_dir)
    index_path = data_dir / 'dataset.index.json'
    manifest_path = data_dir / 'dataset.manifest.jsonl'
    if not index_path.is_file() or not manifest_path.is_file():
        raise FileNotFoundError(
            'M3 directory must contain dataset.index.json and dataset.manifest.jsonl'
        )
    index = _read_json(index_path)
    if tuple(index.get('feature_shape', FEATURE_SHAPE)) != FEATURE_SHAPE:
        raise ValueError(f'Expected feature shape {FEATURE_SHAPE}; got {index.get("feature_shape")}')
    labels = {int(k): str(v) for k, v in index.get('labels', {}).items()}
    if not labels:
        raise ValueError('M3 index has no labels')

    samples = {split: [] for split in ('train', 'val', 'test')}
    targets = {split: [] for split in ('train', 'val', 'test')}
    shard_cache = {}
    for line_number, line in enumerate(manifest_path.read_text(encoding='utf-8').splitlines(), 1):
        if not line.strip():
            continue
        record = json.loads(line)
        split = str(record.get('subset', '')).lower()
        if split not in samples:
            raise ValueError(f'Manifest line {line_number} has unsupported subset: {split!r}')
        if max_samples_per_split is not None and len(samples[split]) >= max_samples_per_split:
            continue
        shard_name = record['shard']
        if shard_name not in shard_cache:
            shard_path = data_dir / shard_name
            if not shard_path.is_file():
                raise FileNotFoundError(f'Missing M3 shard: {shard_path}')
            with np.load(shard_path, allow_pickle=False) as shard:
                shard_cache[shard_name] = (shard['X'].copy(), shard['y'].copy())
        shard_x, shard_y = shard_cache[shard_name]
        row = int(record['row_in_shard'])
        if row < 0 or row >= len(shard_x):
            raise IndexError(f'M3 row {row} is outside {shard_name}')
        sample = np.asarray(shard_x[row], dtype=np.float32)
        if sample.shape != (60, *FEATURE_SHAPE):
            raise ValueError(f'{shard_name}[{row}] has shape {sample.shape}, expected (60, 110, 3)')
        target = int(shard_y[row])
        if target not in labels:
            raise ValueError(f'M3 target {target} is missing from index labels')
        samples[split].append(sample)
        targets[split].append(target)

    loaded = {}
    loaded_targets = {}
    for split in ('train', 'val', 'test'):
        if not samples[split]:
            raise ValueError(f'M3 dataset has no {split} samples')
        loaded[split] = np.stack(samples[split]).astype(np.float32, copy=False)
        loaded_targets[split] = np.asarray(targets[split], dtype=np.int64)
    return loaded, loaded_targets, labels, index

raw_splits, split_targets, labels, m3_index = load_m3_shards(DATA_DIR, MAX_SAMPLES_PER_SPLIT)
class_ids = sorted(labels)
if class_ids != list(range(len(class_ids))):
    raise ValueError(f'M3 class IDs must be contiguous from zero; got {class_ids[:10]}...')
LABEL_NAMES = [labels[i] for i in class_ids]
print('M3 format:', m3_index.get('format'))
print('Feature shape:', raw_splits['train'].shape[1:])
print('Classes:', len(LABEL_NAMES))
print('Official split sizes:', {k: len(v) for k, v in split_targets.items()})

In [ ]:
# Save the mapping before training so the class IDs used by the checkpoint are explicit.
label_mapping = {
    'id_to_label': {str(i): LABEL_NAMES[i] for i in range(len(LABEL_NAMES))},
    'label_to_id': {name: i for i, name in enumerate(LABEL_NAMES)},
    'num_classes': len(LABEL_NAMES),
    'feature_shape': list(FEATURE_SHAPE),
    'input_size': INPUT_SIZE,
    'target_frames': TARGET_FRAMES,
    'preprocessing_version': PREPROCESSING_VERSION,
    'official_splits': {split: int(len(split_targets[split])) for split in ('train', 'val', 'test')},
}
(OUTPUT_DIR / 'label_mapping.json').write_text(
    json.dumps(label_mapping, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
)
print('Saved:', OUTPUT_DIR / 'label_mapping.json')

## 3. Temporal normalization

M3 stores up to 60 decoded frames with right-zero-padding. Since M3 does not retain source FPS, this preprocessing trims trailing all-zero frames and linearly interpolates the remaining sequence to 60 normalized temporal positions. All-zero clips are repeated rather than discarded. This exact policy is recorded in the checkpoint for inference parity.

In [ ]:
def valid_length(sample):
    active = np.any(np.abs(sample) > 0, axis=(1, 2))
    return int(np.flatnonzero(active)[-1]) + 1 if active.any() else 1

def temporal_normalize(sample, target_frames=TARGET_FRAMES):
    arr = np.asarray(sample, dtype=np.float32)
    if arr.ndim != 3 or arr.shape[1:] != FEATURE_SHAPE:
        raise ValueError(f'sample must have shape (frames, {FEATURE_SHAPE[0]}, {FEATURE_SHAPE[1]})')
    if not np.isfinite(arr).all():
        raise ValueError('sample contains non-finite coordinates')
    length = valid_length(arr)
    sequence = arr[:length]
    if length == 1:
        return np.repeat(sequence, target_frames, axis=0).astype(np.float32, copy=False)
    old_positions = np.linspace(0.0, 1.0, length, dtype=np.float32)
    new_positions = np.linspace(0.0, 1.0, target_frames, dtype=np.float32)
    flat = sequence.reshape(length, -1)
    normalized = np.empty((target_frames, flat.shape[1]), dtype=np.float32)
    for feature_index in range(flat.shape[1]):
        normalized[:, feature_index] = np.interp(
            new_positions, old_positions, flat[:, feature_index]
        )
    return normalized.reshape(target_frames, *FEATURE_SHAPE)

normalized_splits = {
    split: np.stack([temporal_normalize(sample) for sample in raw_splits[split]])
    for split in ('train', 'val', 'test')
}
print('Normalized shapes:', {k: v.shape for k, v in normalized_splits.items()})

## 4. Small BiGRU model and data loaders

Each `(110, 3)` frame is flattened to 330 values, projected to 128 dimensions, processed by a two-layer bidirectional GRU, and mean-pooled over time before classification.

In [ ]:
class LandmarkBiGRU(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.LayerNorm(INPUT_SIZE),
            nn.Linear(INPUT_SIZE, HIDDEN_SIZE),
            nn.GELU(),
        )
        self.gru = nn.GRU(
            input_size=HIDDEN_SIZE,
            hidden_size=HIDDEN_SIZE,
            num_layers=GRU_LAYERS,
            batch_first=True,
            dropout=DROPOUT if GRU_LAYERS > 1 else 0.0,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(HIDDEN_SIZE * 2),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_SIZE * 2, num_classes),
        )

    def forward(self, x):
        x = x.reshape(x.shape[0], x.shape[1], INPUT_SIZE)
        x = self.input_projection(x)
        sequence, _ = self.gru(x)
        return self.classifier(sequence.mean(dim=1))

datasets = {
    split: TensorDataset(
        torch.from_numpy(normalized_splits[split].astype(np.float32, copy=False)),
        torch.from_numpy(split_targets[split].astype(np.int64, copy=False)),
    )
    for split in ('train', 'val', 'test')
}
loaders = {
    split: DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=(split == 'train'), num_workers=0, pin_memory=torch.cuda.is_available())
    for split, dataset in datasets.items()
}
model = LandmarkBiGRU(len(LABEL_NAMES)).to(DEVICE)
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 5. Training loop, validation, and checkpoint export

The best validation top-1 model is written to `bigru_best.pt`. The checkpoint includes the model state, architecture/training config, label names, feature contract, normalization version, and split sizes.

In [ ]:
def batch_metrics(logits, targets, top_k=5):
    top1 = (logits.argmax(dim=1) == targets).float().mean().item()
    k = min(top_k, logits.shape[1])
    top_indices = logits.topk(k, dim=1).indices
    top5 = top_indices.eq(targets.unsqueeze(1)).any(dim=1).float().mean().item()
    return {'top1': top1, 'top5': top5}

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_count = 0
    all_logits, all_targets = [], []
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(inputs)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
        count = targets.shape[0]
        total_loss += loss.item() * count
        total_count += count
        all_logits.append(logits.detach().cpu())
        all_targets.append(targets.detach().cpu())
    logits = torch.cat(all_logits)
    targets = torch.cat(all_targets)
    return {'loss': total_loss / max(total_count, 1), **batch_metrics(logits, targets)}

train_targets = split_targets['train']
class_counts = np.bincount(train_targets, minlength=len(LABEL_NAMES)).astype(np.float32)
class_weights = np.ones_like(class_counts)
present = class_counts > 0
class_weights[present] = 1.0 / np.sqrt(class_counts[present])
if present.any():
    class_weights[present] /= class_weights[present].mean()
criterion = nn.CrossEntropyLoss(weight=torch.from_numpy(class_weights).to(DEVICE))
eval_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

config = {
    'input_size': INPUT_SIZE, 'feature_shape': list(FEATURE_SHAPE), 'target_frames': TARGET_FRAMES,
    'hidden_size': HIDDEN_SIZE, 'gru_layers': GRU_LAYERS, 'dropout': DROPOUT,
    'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'patience': PATIENCE, 'grad_clip': GRAD_CLIP,
}
best_val_top1 = -1.0
best_epoch = 0
stale_epochs = 0
history = []
for epoch in range(1, EPOCHS + 1):
    started = time.time()
    train_metrics = run_epoch(model, loaders['train'], criterion, optimizer)
    val_metrics = run_epoch(model, loaders['val'], eval_criterion)
    scheduler.step(val_metrics['top1'])
    row = {'epoch': epoch, 'train': train_metrics, 'val': val_metrics, 'lr': optimizer.param_groups[0]['lr']}
    history.append(row)
    print(
        f"epoch {epoch:03d}: train loss={train_metrics['loss']:.4f} top1={train_metrics['top1']:.3f} | "
        f"val loss={val_metrics['loss']:.4f} top1={val_metrics['top1']:.3f} top5={val_metrics['top5']:.3f} "
        f"({time.time() - started:.1f}s)"
    )
    if val_metrics['top1'] > best_val_top1:
        best_val_top1 = val_metrics['top1']
        best_epoch = epoch
        stale_epochs = 0
        checkpoint = {
            'model_state': model.state_dict(),
            'config': config,
            'label_names': LABEL_NAMES,
            'feature_shape': FEATURE_SHAPE,
            'max_frames': TARGET_FRAMES,
            'preprocessing_version': PREPROCESSING_VERSION,
            'target_mobile_fps': 15,
            'seed': SEED,
            'best_val_top1': best_val_top1,
        }
        torch.save(checkpoint, OUTPUT_DIR / 'bigru_best.pt')
        print('  saved new best checkpoint')
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f'Early stopping after {epoch} epochs')
            break

if not (OUTPUT_DIR / 'bigru_best.pt').is_file():
    raise RuntimeError('Training did not produce a checkpoint')
print('Best epoch:', best_epoch, 'validation top-1:', best_val_top1)

## 6. Test evaluation and saved metrics

Reloading the best checkpoint makes this evaluation independent of the final in-memory epoch. The report includes top-1/top-5 metrics and official split counts.

In [ ]:
best_checkpoint = torch.load(OUTPUT_DIR / 'bigru_best.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(best_checkpoint['model_state'])
test_metrics = run_epoch(model, loaders['test'], eval_criterion)
report = {
    'device': str(DEVICE),
    'classes': len(LABEL_NAMES),
    'label_names': LABEL_NAMES,
    'best_epoch': best_epoch,
    'best_val_top1': best_val_top1,
    'test': test_metrics,
    'history': history,
    'preprocessing_version': PREPROCESSING_VERSION,
    'feature_shape': list(FEATURE_SHAPE),
    'input_size': INPUT_SIZE,
    'target_mobile_fps': 15,
    'official_splits': {split: len(split_targets[split]) for split in ('train', 'val', 'test')},
}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(f"test loss={test_metrics['loss']:.4f} top1={test_metrics['top1']:.3f} top5={test_metrics['top5']:.3f}")
print('Saved checkpoint:', OUTPUT_DIR / 'bigru_best.pt')
print('Saved metrics:', OUTPUT_DIR / 'metrics.json')
print('Saved labels:', OUTPUT_DIR / 'label_mapping.json')

In [ ]:
# Optional training curves and a compact classification report.
try:
    from sklearn.metrics import classification_report
    all_predictions, all_targets = [], []
    model.eval()
    with torch.no_grad():
        for inputs, targets in loaders['test']:
            all_predictions.extend(model(inputs.to(DEVICE)).argmax(dim=1).cpu().numpy())
            all_targets.extend(targets.numpy())
    print(classification_report(
        all_targets, all_predictions, labels=list(range(len(LABEL_NAMES))),
        target_names=LABEL_NAMES, zero_division=0, digits=3
    ))
except ImportError:
    print('Install scikit-learn to display the per-class report.')

try:
    import matplotlib.pyplot as plt
    epochs_seen = [row['epoch'] for row in history]
    plt.plot(epochs_seen, [row['train']['loss'] for row in history], label='train loss')
    plt.plot(epochs_seen, [row['val']['loss'] for row in history], label='val loss')
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.2); plt.show()
except ImportError:
    print('Install matplotlib to display training curves.')

### Exported artifacts

The output directory contains:
- `bigru_best.pt`: model state plus input/normalization/label metadata;
- `metrics.json`: validation history, test top-1/top-5, and official split sizes;
- `label_mapping.json`: stable class-ID ↔ WLASL gloss mapping required by inference.

For mobile/inference parity, reuse `temporal_normalize` with `(110, 3)` frames and the `temporal-normalize-nonzero-v1` policy recorded above.